<h1>✅ LÝ THUYẾT VỀ PINECONE</h1>

<h3>📌1. Pinecone là gì?</h3>

Pinecone là một Vector Database chuyên biệt, được thiết kế để:

+ Lưu trữ các vector embedding

+ Hỗ trợ truy vấn tương đồng (similarity search) rất nhanh và hiệu quả

+ Tối ưu cho các ứng dụng AI hiện đại như:

    + Chatbot tích hợp RAG

    + Search thông minh (semantic search)

    + Recommendation systems

<h3>📚2. Vector là gì trong Pinecone?</h3>

+ Là dạng số hóa của dữ liệu phi cấu trúc (văn bản, hình ảnh, âm thanh, mã nguồn…)

+ Được trích xuất bởi các mô hình embedding (ví dụ: OpenAI, BERT, Sentence Transformers, Cohere...)

+ Ví dụ: một câu văn sau khi embedding sẽ thành 1 vector 768 hoặc 1536 chiều:

In [ ]:
[0.01, 0.82, -0.23, ..., 0.42]  # dạng float

<h3>🎯3. Pinecone dùng để làm gì?</h3>

| Use case                    | Mô tả                                                |
| --------------------------- | ---------------------------------------------------- |
| Chatbot + RAG               | Truy xuất các đoạn văn bản liên quan → đưa vào GPT   |
| Semantic Search             | Tìm kiếm theo **ý nghĩa**, không chỉ từ khóa         |
| Personalized Recommendation | Tìm sản phẩm tương đồng theo vector người dùng       |
| Anomaly Detection           | Phát hiện dữ liệu bất thường theo khoảng cách vector |


<h3>⚙️4. Các thành phần chính trong Pinecone</h3>

| Thành phần            | Mô tả                                                            |
| --------------------- | ---------------------------------------------------------------- |
| **Index**             | Kho lưu trữ vector, bạn có thể tạo nhiều index                   |
| **Vector**            | Một embedding cụ thể gồm: `id`, `values`, và optional `metadata` |
| **Namespace**         | Không gian con trong 1 index, tiện phân loại                     |
| **Similarity Metric** | Cách tính độ tương đồng: `cosine`, `dot product`, `euclidean`    |

<h3>🧠5. Cách hoạt động của Pinecone (trong RAG)</h3>

1. Tách tài liệu thành các đoạn nhỏ (chunks)

2. Embed các đoạn → vector

3. Lưu vector vào Pinecone

4. Khi người dùng đặt câu hỏi:

    + Embed câu hỏi → vector

    + Tìm các vector gần nhất trong Pinecone

    + Lấy văn bản tương ứng → đưa vào prompt của GPT

<h1>✅ CÁCH SỬ DỤNG PINECONE</h1>

<h3>1. Cài đặt thư viện</h3>

In [1]:
! pip install -U pinecone-client

  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)


In [2]:
! pip install pinecone

  Using cached pinecone_plugin_assistant-1.7.0-py3-none-any.whl.metadata (28 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 5.4 MB/s eta 0:00:00
Using cached pinecone_plugin_assistant-1.7.0-py3-none-any.whl (239 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 23.2
    Uninstalling packaging-23.2:
      Successfully uninstalled packaging-23.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.1.53 requires packaging<24.0,>=23.2, but you have packaging 24.2 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


<h3>2. Đăng ký Pinecone & lấy API Key</h3>

+ Truy cập: https://app.pinecone.io/

+ Tạo tài khoản

+ Tạo API Key và ghi lại Environment (ví dụ: gcp-starter)

<h3>3. Khởi tạo Pinecone client</h3>

In [3]:
import pinecone

In [ ]:
pinecone.init(
    api_key = "YOUR_PINECONE_API_KEY",
    environment = "gpc-starter" # hoac "us-east1-gcp"
)

<h3>4. Tạo index</h3>

In [ ]:
pinecone.create_index("my-index", dimension=1538, metric="cosine")
index = pinecone.Index("my-index")

Ghi chú: 1536 là số chiều embedding từ mô hình text-embedding-3-small của OpenAI.

<h3>5. Nhúng (embed) và lưu dữ liệu</h3>

In [ ]:
from openai import OpenAI
OpenAI.api_key = "YOUR_OPENAI_API_KEY"

def embed_text(text):
    res = OpenAI.Embedding.create(
        model="text-embedding-3-small",
        input=text
    )
    return OpenAI["data"][0]["embedding"]


# Thêm vector embedding vào index
texts = ["Chính sách bảo hành sản phẩm", "Hướng dẫn sử dụng thiết bị", "Cách đặt lịch sửa chữa"]
ids = ["doc1", "doc2", "doc3"]

vectors=[(ids[1], embed_text(texts[i])) for i in range(len(texts))]
index.upsert(vectors)

<h3>6. Truy vấn vector tương tự</h3>

In [ ]:
query = "Bảo hành kéo dài trong bao lâu?"
query_vector = embed_text(query)

results = index.query(vector=query_vector, top_k=3, include_metadata=False)

for match in results["matches"]:
    print(f"ID: {match['id']}, Score: {match['score']}")

<h3>7. Xoá / quản lý dữ liệu</h3>

In [ ]:
index.delete(ids=["doc1"])

pinecone.delete_index(index_name)

<h1>📦 Tổng kết: Pinecone là gì và dùng thế nào?</h1>

| Mục            | Nội dung                                                            |
| -------------- | ------------------------------------------------------------------- |
| Pinecone là gì | Vector DB lưu trữ embedding và cho phép tìm kiếm tương tự           |
| Dùng để làm gì | Chatbot RAG, Semantic Search, đề xuất cá nhân hóa                   |
| Cách dùng      | 1. Embed dữ liệu → 2. Lưu vào Pinecone → 3. Truy vấn khi có câu hỏi |
| Ưu điểm        | Nhanh, tối ưu tìm kiếm vector, tích hợp dễ với OpenAI, LangChain    |
